# 01 — Full Pipeline (Discovery → Validation → Catalog)
Part of the Public Webcam Discovery System.

Runs the four core pipeline stages in sequence:
1. **DirectoryAgent** — crawls Tier 1 sources from `SOURCES.md` → `candidates/candidates.jsonl`
2. **SearchAgent** — structured multi-language DuckDuckGo queries → `candidates/search_candidates.jsonl` (merged into `candidates/candidates.jsonl`)
3. **ValidationAgent** — probes each URL for live HLS/MJPEG streams → `candidates/validated.jsonl`
4. **CatalogAgent** — deduplicates, geo-enriches, and exports → `camera.geojson` + `cameras.md`

In [ ]:
# Environment setup — run this cell first every session
import subprocess, sys, os
from pathlib import Path

IN_COLAB     = "google.colab" in sys.modules
IN_SAGEMAKER = os.environ.get("SM_TRAINING_ENV") is not None

# ── 1. Locate / clone the project ────────────────────────────────────────────
if IN_COLAB:
    repo_dir = Path("/content/webcam-discovery")
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
elif IN_SAGEMAKER:
    repo_dir = Path(os.environ.get("SM_MODEL_DIR", "/opt/ml/model")) / "webcam-discovery"
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/dshipley71/webcam-discovery", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
else:
    # Local: assume CWD is already the project root (where pyproject.toml lives)
    repo_dir = Path.cwd()

print(f"Project root: {repo_dir}")

# ── 2. Install the package using the absolute path ───────────────────────────
# Using repo_dir directly avoids any CWD ambiguity (relative "." can resolve to
# the wrong directory if the kernel was restarted or chdir ran more than once).
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{repo_dir}[notebooks]", "-q"],
    check=True,
)

# ── 3. Verify importability — fallback: insert src/ into sys.path ─────────────
# pip install -e works in a fresh kernel but can be lost after a kernel restart
# if the site-packages link is stale.  Inserting src/ ensures the package is
# always resolvable without a second install.
src_path = str(repo_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import webcam_discovery  # noqa: F401  — raises ImportError if src layout is broken
from webcam_discovery.config import settings
settings.ensure_dirs()

print(f"webcam_discovery loaded from: {Path(webcam_discovery.__file__).parent}")
print("Ready ✓")

## Step 2 - Build environment

Install the package with dev dependencies so the pipeline and tests are available.

In [ ]:
import subprocess, sys
from pathlib import Path

# repo_dir is set by the env setup cell above.
# Fallback to CWD so this cell is also safe to run standalone.
try:
    _repo = repo_dir  # type: ignore[name-defined]
except NameError:
    _repo = Path.cwd()

# Install dev extras (pytest, respx, etc.) using the same absolute-path pattern
# as the env setup cell to avoid CWD ambiguity.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", f"{_repo}[dev]", "-q"],
    check=True,
)
print("Dev dependencies installed ✓")

## Step 3a — Run DirectoryAgent (Tier 1)

Crawls all Tier 1 sources from `SOURCES.md` and resolves direct `.m3u8`/`.mjpeg`
stream URLs via `FeedExtractionSkill`. Results are written to `candidates/candidates.jsonl`.

### Crawler behaviour

- **`DirectoryTraversalSkill`** — follows sub-category links up to `max_depth=5` to reach
  individual camera pages (e.g. `/en/usa/new-york/niagara.html` is 4 segments deep)
- **`FeedExtractionSkill`** — scans static HTML for `.m3u8`/`.mjpeg` URLs including
  JSON blobs, `window.__data__` assignments, and non-standard variable names
- **robots.txt checks run concurrently** — all source domains are checked in parallel
  before crawling begins, so blocked domains add no serial latency
- `MAX_PAGES_PER_SOURCE = 100` guard prevents runaway crawls

In [4]:
!python scripts/run_discovery.py --tier 1 --output candidates/candidates.jsonl

2026-03-13 18:42:47.238 | INFO     | webcam_discovery.agents.directory_crawler:_parse:142 - SourcesRegistry: loaded tiers {1: 14, 2: 16, 3: 24, 4: 7, 5: 2} (63 total sources), 12 blocked domains from SOURCES.md
2026-03-13 18:42:47.238 | INFO     | webcam_discovery.agents.directory_crawler:run:222 - DirectoryAgent: tier=1 → 14 sources across 1 tier(s)
2026-03-13 18:42:47.904 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows indy.com
2026-03-13 18:42:48.039 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed for ebcamtaxi.com: [Errno -2] Name or service not known
2026-03-13 18:42:48.040 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows ebcamtaxi.com
2026-03-13 18:42:48.505 | DEBUG    | webcam_discovery.agents.directory_crawler:run:245 - DirectoryAgent: robots.txt allows skylinewebcams.com
2026-03-13 18:42:48.749 | DEBUG    | webcam_discovery.agents.directory_cra

## Step 3b — Run SearchAgent (DuckDuckGo discovery)

`SearchAgent` complements directory crawling with structured multi-language queries
against DuckDuckGo (no API key required). It generates queries for each Tier-N city
in English and other languages, filtering out blocked domains and known non-HLS sources.

Results are written to `candidates/search_candidates.jsonl`, then merged (deduplicated
by URL) with the directory-crawl results into `candidates/candidates.jsonl`.

### Search behaviour

- **`QueryGenerationSkill`** — generates query templates in English, Japanese, German,
  French, Spanish, Korean, Swedish, and Norwegian for each city
- **DuckDuckGo HTML endpoint** — no API key, polite 0.5 s delay between requests,
  concurrency capped at 5 simultaneous searches
- **`MAX_QUERIES_PER_CITY = 3`** guard avoids rate-limit triggers
- Blocked domains (e.g. `insecam.org`, `shodan.io`) and non-HLS source domains
  are filtered out before results are written

In [ ]:
!python scripts/run_search.py --tier 1 --output candidates/search_candidates.jsonl

In [ ]:
import json
from pathlib import Path

dir_path    = Path("candidates/candidates.jsonl")
search_path = Path("candidates/search_candidates.jsonl")
merged_path = Path("candidates/candidates.jsonl")

dir_lines    = dir_path.read_text().splitlines()    if dir_path.exists()    else []
search_lines = search_path.read_text().splitlines() if search_path.exists() else []

seen_urls: set[str] = set()
merged_lines: list[str] = []
for line in dir_lines + search_lines:
    if not line.strip():
        continue
    try:
        url = json.loads(line).get("url", "")
        if url and url not in seen_urls:
            seen_urls.add(url)
            merged_lines.append(line)
    except json.JSONDecodeError:
        continue

merged_path.write_text("\n".join(merged_lines))
print(f"Directory crawl  : {len(dir_lines)} candidates")
print(f"Search results   : {len(search_lines)} candidates")
print(f"Merged (deduped) : {len(merged_lines)} candidates → {merged_path}")

## Step 4 — Inspect candidates.jsonl

Preview candidate counts broken down by URL type.

The crawler outputs a mix of direct stream URLs (when found in static HTML) and
camera page URLs (when the stream URL requires JavaScript or further probing). The
`ValidationAgent` in Step 5 handles both: it confirms HLS/MJPEG stream URLs
directly, and probes HTML/embed pages for embedded stream URLs.

| Type | Meaning | Handling |
|------|---------|----------|
| `hls-stream`    | `.m3u8` HLS playlist URL             | ✅ Confirmed by validator |
| `mjpeg-stream`  | `.mjpg` / `.mjpeg` MJPEG stream URL  | ✅ Confirmed by validator |
| `html-page`     | Camera page (stream URL not in HTML) | 🔍 Validator probes for embedded stream |
| `embed-page`    | Third-party player iframe URL        | 🔍 Validator probes for embedded stream |
| `youtube-embed` | YouTube nocookie embed URL           | ❌ Rejected by validator (not a stream) |
| `mp4-file`      | Static MP4 video recording           | ❌ Rejected by validator (not live) |

In [ ]:
import json, re
from pathlib import Path
from collections import Counter

output_path = Path("candidates/candidates.jsonl")

if not output_path.exists():
    print("candidates.jsonl not found - did the crawler run successfully?")
else:
    lines = output_path.read_text().splitlines()
    candidates = [json.loads(l) for l in lines if l.strip()]

    # HLS (.m3u8) and MJPEG (.mjpg/.mjpeg) are confirmed stream URLs.
    # HTML and embed pages are still valid — validator will probe them for embedded streams.
    _hls_re    = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)
    _mjpeg_re  = re.compile(r'\.(mjpg|mjpeg)(\?|$)', re.IGNORECASE)
    _mp4_re    = re.compile(r'\.(mp4|webm|ogv)(\?|$)', re.IGNORECASE)
    _youtube_re = re.compile(r'youtube(?:-nocookie)?\.com/embed/', re.IGNORECASE)
    _embed_re   = re.compile(r'/embed[/\?]|embed\.', re.IGNORECASE)

    def url_type(url):
        if _hls_re.search(url):     return 'hls-stream'
        if _mjpeg_re.search(url):   return 'mjpeg-stream'
        if _mp4_re.search(url):     return 'mp4-file'
        if _youtube_re.search(url): return 'youtube-embed'
        if _embed_re.search(url):   return 'embed-page'
        return 'html-page'

    types   = Counter(url_type(c['url']) for c in candidates)
    sources = Counter(c.get('source_directory', 'unknown') for c in candidates)
    valid   = types['hls-stream'] + types['mjpeg-stream']

    # Distinguish directory-crawl vs SearchAgent sources
    search_count = sum(n for s, n in sources.items() if s.startswith('search:'))
    crawl_count  = len(candidates) - search_count

    print(f"Total candidates: {len(candidates)}")
    print(f"  From directory crawl : {crawl_count}")
    print(f"  From search results  : {search_count}")
    print()
    print("-- URL type breakdown --")
    print(f"  hls-stream     : {types['hls-stream']:>4}  ✅ (.m3u8 HLS playlist — confirmed stream)")
    print(f"  mjpeg-stream   : {types['mjpeg-stream']:>4}  ✅ (.mjpeg MJPEG stream — confirmed stream)")
    print(f"  embed-page     : {types['embed-page']:>4}  🔍 (embed page — validator will probe for stream)")
    print(f"  html-page      : {types['html-page']:>4}  🔍 (camera page — validator will probe for stream)")
    print(f"  mp4-file       : {types['mp4-file']:>4}  ❌ (static video, rejected by validator)")
    print(f"  youtube-embed  : {types['youtube-embed']:>4}  ❌ (not a live stream, rejected by validator)")
    print(f"\n  Already confirmed stream URLs : {valid}")
    print(f"  Pending HTML probe by validator: {types['embed-page'] + types['html-page']}")
    print(f"  Will be rejected              : {types['mp4-file'] + types['youtube-embed']}")
    print()
    print("-- Top source directories --")
    for src, n in sources.most_common(15):
        print(f"  {n:>4}  {src}")
    print()
    print("-- First 10 candidates --")
    for c in candidates[:10]:
        t = url_type(c['url'])
        mark = '✅' if t in ('hls-stream', 'mjpeg-stream') else ('🔍' if t in ('embed-page', 'html-page') else '❌')
        print(f"  {mark} [{t:<14}]  {c['url']}")
        print(f"               label={c.get('label','-')}  city={c.get('city','-')}, {c.get('country','-')}")
        print()

## Step 5 - Validate candidates

`ValidationAgent` deep-probes each URL to confirm it is an **active live stream**.

### Stream-only validation policy

Every output URL must be a directly playable stream — no HTML pages, no YouTube
embeds, no MP4 recordings.

| URL type | Probe strategy | Outcome |
|----------|----------------|---------|
| `.m3u8`  | GET first 1 KB; verify `#EXTM3U` magic | ✅ high/live if magic found |
| `.mjpeg` | HEAD for `multipart/x-mixed-replace`; byte-probe fallback | ✅ high/live if confirmed |
| YouTube embed | Immediate reject | ❌ dead (not a stream) |
| MP4 / WebM | Immediate reject | ❌ dead (static video, not live) |
| HTML page | GET + scan for embedded `.m3u8` / `.mjpeg` URLs; probe each | ✅ high/live only if a stream URL inside is confirmed; ❌ dead otherwise |

### Geocoding fallback chain

1. City + country via Nominatim
2. Label text via Nominatim (e.g. "Times Square NYC")
3. IP geolocation of camera hostname via ip-api.com
4. Country-center via Nominatim

### Legitimacy grades (stream URLs only)

| Grade | Criteria |
|-------|----------|
| high  | HLS `#EXTM3U` magic confirmed, or MJPEG `multipart/x-mixed-replace` confirmed |
| high  | Stream URL extracted from HTML page and confirmed live |

Results → `candidates/validated.jsonl`

In [6]:
!python scripts/run_validation.py --input candidates/candidates.jsonl --output candidates/validated.jsonl

2026-03-13 18:52:59.111 | INFO     | webcam_discovery.agents.validator:run:107 - ValidationAgent: 646 candidates received
2026-03-13 18:53:00.543 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed for : Request URL is missing an 'http://' or 'https://' protocol.
2026-03-13 18:53:03.058 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed for orldcam.eu: [Errno -2] Name or service not known
2026-03-13 18:53:03.068 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed for youtube-nocookie.com: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: Hostname mismatch, certificate is not valid for 'youtube-nocookie.com'. (_ssl.c:1010)
2026-03-13 18:53:03.092 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed for ebcamera24.com: [Errno -2] Name or service not known
2026-03-13 18:53:03.107 | WARNING  | webcam_discovery.skills.validation:run:456 - robots.txt fetch failed 

## Step 6 - Inspect validated cameras

Pipeline funnel, feed-type breakdown, and all confirmed active stream records.
Every URL in the output must be a directly playable HLS or MJPEG stream.

In [ ]:
import json, re
from pathlib import Path
from collections import Counter

validated_path  = Path("candidates/validated.jsonl")
candidates_path = Path("candidates/candidates.jsonl")

if not validated_path.exists():
    print("validated.jsonl not found - did validation run successfully?")
else:
    records = [json.loads(l) for l in validated_path.read_text().splitlines() if l.strip()]
    n_cands = sum(1 for l in candidates_path.read_text().splitlines() if l.strip()) \
              if candidates_path.exists() else '?'

    located   = [r for r in records if r.get('latitude') is not None]
    unlocated = [r for r in records if r.get('latitude') is None]

    # Valid FeedType literals from schemas.py:
    #   "HLS_master" — master playlist with #EXT-X-STREAM-INF variant references
    #   "HLS_stream" — media playlist with #EXTINF segments (direct live stream)
    #   "MJPEG"      — multipart/x-mixed-replace or .mjpg/.mjpeg URL
    _VALID_FEED_TYPES = {'HLS_master', 'HLS_stream', 'MJPEG'}

    _hls_re   = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)
    _mjpeg_re = re.compile(r'\.(mjpg|mjpeg)(\?|$)', re.IGNORECASE)

    def stream_category(r):
        url = r.get('url', '')
        ft  = r.get('feed_type', '')
        if _hls_re.search(url) or ft in ('HLS_master', 'HLS_stream'): return 'hls-stream ✅'
        if _mjpeg_re.search(url) or ft == 'MJPEG':                    return 'mjpeg-stream ✅'
        return f'non-stream ❌ [{ft}]'

    categories = Counter(stream_category(r) for r in records)
    valid_count   = sum(n for k, n in categories.items() if '✅' in k)
    invalid_count = sum(n for k, n in categories.items() if '❌' in k)

    print("-- Pipeline funnel ------------------------------------------")
    print(f"  Candidates (discovery + search) : {n_cands}")
    print(f"  Validated records               : {len(records)}")
    print(f"    with coordinates              : {len(located)}")
    print(f"    without coords (included)     : {len(unlocated)}")
    print(f"    active stream URLs            : {valid_count}  ✅")
    print(f"    non-stream URLs (bug!)        : {invalid_count}  ❌")
    print()
    if invalid_count > 0:
        print("  ⚠️  Non-stream records found — check validation logic.")
        print()

    print("-- Stream type breakdown ------------------------------------")
    for cat, n in categories.most_common():
        print(f"  {cat:<30} : {n}")
    print()
    print("-- By feed type ----------------------------------------------")
    for ft, n in Counter(r['feed_type'] for r in records).most_common():
        mark = '✅' if ft in _VALID_FEED_TYPES else ('⚠️' if ft == 'unknown' else '❌')
        print(f"  {mark} {ft:<16} : {n}")
    print()
    print("-- By legitimacy --------------------------------------------")
    for ls, n in Counter(r['legitimacy_score'] for r in records).most_common():
        print(f"  {ls:<8} : {n}")
    print()
    print("-- All validated stream records (highest legitimacy first) --")
    top = sorted(records,
                 key=lambda r: {'high': 2, 'medium': 1, 'low': 0}.get(r['legitimacy_score'], 0),
                 reverse=True)
    for r in top:
        coords = f"{r['latitude']:.3f},{r['longitude']:.3f}" if r.get('latitude') else 'no-coords'
        ft     = r['feed_type']
        mark   = '✅' if ft in _VALID_FEED_TYPES else '❌'
        print(f"  {mark} [{ft:<14}] [{r['legitimacy_score']:<6}] [{coords}]")
        print(f"    {r['url']}")
        print(f"    {r.get('label', '-')} — {r.get('city', '-')}, {r.get('country', '-')}")
        print()

## Step 7 — Build catalog (CatalogAgent)

`CatalogAgent` takes the validated records and:
1. **Deduplicates** — merges records with identical or near-identical stream URLs
2. **Exports** `camera.geojson` — GeoJSON FeatureCollection (RFC 7946 `[lon, lat]`)
3. **Exports** `cameras.md` — human-readable camera link list
4. **Snapshots** — writes a dated copy to `snapshots/camera_YYYY-MM-DD.geojson`

Both output files are written to the project root so that `map.html` can auto-load `camera.geojson`.

In [ ]:
!python scripts/run_catalog.py --input candidates/validated.jsonl --output .

## Step 8 — Inspect catalog outputs

Summary of `camera.geojson` features, coordinate coverage, and a preview of `cameras.md`.
Serve `map.html` locally to view the interactive map (camera.geojson must be co-located).

In [ ]:
import json
from pathlib import Path
from collections import Counter

geojson_path = Path("camera.geojson")
md_path      = Path("cameras.md")

if not geojson_path.exists():
    print("camera.geojson not found — did the catalog step run successfully?")
else:
    gj = json.loads(geojson_path.read_text())
    features = gj.get("features", [])

    located   = [f for f in features if f["geometry"] is not None]
    unlocated = [f for f in features if f["geometry"] is None]
    countries = Counter(
        f["properties"].get("country", "unknown") for f in features
    )
    feed_types = Counter(
        f["properties"].get("feed_type", "unknown") for f in features
    )
    sources = Counter(
        f["properties"].get("source_directory", "unknown") for f in features
    )

    print("── camera.geojson ───────────────────────────────────────")
    print(f"  Total features          : {len(features)}")
    print(f"  With coordinates        : {len(located)}")
    print(f"  Without coordinates     : {len(unlocated)}")
    print()
    print("── Feed types ───────────────────────────────────────────")
    for ft, n in feed_types.most_common():
        print(f"  {ft:<20} : {n}")
    print()
    print("── Top countries ────────────────────────────────────────")
    for country, n in countries.most_common(15):
        print(f"  {country:<30} : {n}")
    print()
    print("── Top source directories ───────────────────────────────")
    for src, n in sources.most_common(10):
        print(f"  {n:>4}  {src}")
    print()
    print("── First 5 features ─────────────────────────────────────")
    for f in features[:5]:
        p = f["properties"]
        geo = f["geometry"]
        coords = f"{geo['coordinates'][1]:.3f},{geo['coordinates'][0]:.3f}" if geo else "no-coords"
        print(f"  [{p.get('feed_type','?'):<14}] [{coords}]")
        print(f"    {p.get('label','-')} — {p.get('city','-')}, {p.get('country','-')}")
        print(f"    {p.get('url','')}")
        print()

    if md_path.exists():
        lines = md_path.read_text().splitlines()
        print(f"── cameras.md ({len(lines)} lines) — first 20 ──────────────────")
        print("\n".join(lines[:20]))

    print()
    print("── To view the interactive map ──────────────────────────")
    print("  python -m http.server 8000")
    print("  open http://localhost:8000/map.html")

## Step 9 — Export intermediate datasets as GeoJSON

Three supplementary GeoJSON exports for debugging and cross-pipeline comparison.
All three reuse `DeduplicationSkill` and `GeoJSONExportSkill` from the catalog layer.

| Output file | Source | Filter | Notes |
|-------------|--------|--------|-------|
| `validated.geojson` | `candidates/validated.jsonl` | all records | Deduped CameraRecord objects with full geo-enrichment |
| `candidates.geojson` | `candidates/candidates.jsonl` | HLS + MJPEG only | Raw discovery candidates geo-enriched on the fly |
| `search_candidates.geojson` | `candidates/search_candidates.jsonl` | HLS + MJPEG only | Search candidates geo-enriched on the fly |

**Why filter candidates to HLS/MJPEG?** Raw candidates include HTML embed pages and
other non-stream URLs that only become mappable after the validator probes them.
Filtering here ensures every feature in these GeoJSONs has a confirmed direct stream URL.

> **Cells 9b and 9c use `await`** — this requires IPython ≥ 7 (Jupyter Lab, Jupyter Notebook 6+,
> Google Colab) which enables top-level `await` inside notebook cells.

In [ ]:
"""
Step 9a — Export validated.geojson.

Loads candidates/validated.jsonl (full CameraRecord objects produced by
ValidationAgent), deduplicates using the same DeduplicationSkill that
CatalogAgent uses, then exports to validated.geojson via GeoJSONExportSkill.

This gives a GeoJSON snapshot of every record that passed validation before
the final catalog deduplication step.
"""
import json
from pathlib import Path
from webcam_discovery.schemas import CameraRecord
from webcam_discovery.skills.catalog import (
    DeduplicationSkill, DeduplicationInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

input_path  = Path("candidates/validated.jsonl")
output_path = Path("validated.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 5 first")
else:
    records = [
        CameraRecord(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]
    print(f"Loaded {len(records)} validated records")

    # Deduplicate using the same skill CatalogAgent uses
    dedup_skill = DeduplicationSkill()
    catalog: list[CameraRecord] = []
    for record in records:
        result = dedup_skill.run(DeduplicationInput(
            candidate_record=record,
            existing_catalog=catalog,
        ))
        if result.is_duplicate:
            if result.merged_record and result.canonical_record:
                idx = next(
                    (i for i, r in enumerate(catalog) if r.id == result.canonical_record.id),
                    None,
                )
                if idx is not None:
                    catalog[idx] = result.merged_record
        else:
            catalog.append(record)

    print(f"After dedup: {len(catalog)} records ({len(records) - len(catalog)} duplicates removed)")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=catalog, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")

In [ ]:
"""
Step 9b — Export candidates.geojson (HLS + MJPEG only).

Filters candidates.jsonl to direct HLS (.m3u8) and MJPEG (.mjpg/.mjpeg)
stream URLs — those already resolved to a live stream URL by FeedExtractionSkill
during directory crawling.  HTML/embed pages are excluded here (they are handled
by the validator in Step 5).

Uses GeoEnrichmentSkill to add coordinates from candidate city/country metadata,
then exports via GeoJSONExportSkill.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE   = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)
_MJPEG_RE = re.compile(r'\.(mjpg|mjpeg)(\?|$)', re.IGNORECASE)

def _classify_url(url: str):
    if _HLS_RE.search(url):   return "HLS_stream"
    if _MJPEG_RE.search(url): return "MJPEG"
    return None

input_path  = Path("candidates/candidates.jsonl")
output_path = Path("candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3a first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    stream_cands = [(c, _classify_url(c.url)) for c in all_cands]
    stream_cands = [(c, ft) for c, ft in stream_cands if ft is not None]
    print(f"candidates.jsonl : {len(all_cands)} total → {len(stream_cands)} HLS/MJPEG streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(pairs):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c, _ in pairs
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(stream_cands)

    records: list[CameraRecord] = []
    for (candidate, feed_type), geo in zip(stream_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = feed_type,
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")

In [ ]:
"""
Step 9c — Export search_candidates.geojson (HLS + MJPEG only).

Filters search_candidates.jsonl to direct HLS (.m3u8) and MJPEG (.mjpg/.mjpeg)
stream URLs, geo-enriches each record using GeoEnrichmentSkill (city + country
fallback chain), then exports via GeoJSONExportSkill.

Note: SearchAgent returns DuckDuckGo result pages, so most URLs are HTML pages
rather than direct stream links.  HLS/MJPEG hits here are rare but valuable.
"""
import asyncio, json, re
from pathlib import Path
from slugify import slugify
from webcam_discovery.schemas import CameraCandidate, CameraRecord
from webcam_discovery.skills.catalog import (
    GeoEnrichmentSkill, GeoEnrichmentInput,
    GeoJSONExportSkill, GeoJSONExportInput,
)

_HLS_RE   = re.compile(r'\.m3u8(\?|$)', re.IGNORECASE)
_MJPEG_RE = re.compile(r'\.(mjpg|mjpeg)(\?|$)', re.IGNORECASE)

def _classify_url(url: str):
    if _HLS_RE.search(url):   return "HLS_stream"
    if _MJPEG_RE.search(url): return "MJPEG"
    return None

input_path  = Path("candidates/search_candidates.jsonl")
output_path = Path("search_candidates.geojson")

if not input_path.exists():
    print(f"{input_path} not found — run Step 3b first")
else:
    all_cands = [
        CameraCandidate(**json.loads(line))
        for line in input_path.read_text().splitlines()
        if line.strip()
    ]

    stream_cands = [(c, _classify_url(c.url)) for c in all_cands]
    stream_cands = [(c, ft) for c, ft in stream_cands if ft is not None]
    print(f"search_candidates.jsonl : {len(all_cands)} total → {len(stream_cands)} HLS/MJPEG streams")

    geo_skill = GeoEnrichmentSkill()

    async def _enrich_all(pairs):
        tasks = [
            geo_skill.run(GeoEnrichmentInput(
                city    = c.city    if c.city    and c.city    != "Unknown" else None,
                country = c.country if c.country and c.country != "Unknown" else None,
                label   = c.label   if c.label   and c.label   != c.city    else None,
                url     = c.url,
            ))
            for c, _ in pairs
        ]
        return await asyncio.gather(*tasks, return_exceptions=True)

    geo_results = await _enrich_all(stream_cands)

    records: list[CameraRecord] = []
    for (candidate, feed_type), geo in zip(stream_cands, geo_results):
        city    = candidate.city    or "Unknown"
        country = candidate.country or "Unknown"
        label   = candidate.label   or city
        lat = lon = continent = region = None
        if isinstance(geo, Exception):
            pass
        elif geo is not None:
            lat, lon  = geo.latitude, geo.longitude
            continent = geo.continent or "Unknown"
            region    = geo.region
            if geo.country:
                country = geo.country

        records.append(CameraRecord(
            id               = slugify(f"{city} {label}", max_length=80, word_boundary=True, separator="-") or "camera",
            label            = label,
            city             = city,
            country          = country,
            continent        = continent or "Unknown",
            region           = region,
            latitude         = lat,
            longitude        = lon,
            url              = candidate.url,
            feed_type        = feed_type,
            source_directory = candidate.source_directory,
            source_refs      = candidate.source_refs or [candidate.url],
            legitimacy_score = "low",
            status           = "unknown",
            notes            = candidate.notes,
        ))

    located = sum(1 for r in records if r.latitude is not None)
    print(f"Geo-enriched: {located}/{len(records)} records have coordinates")

    result = GeoJSONExportSkill().run(GeoJSONExportInput(cameras=records, output_path=output_path))
    print(f"Exported : {result.exported} features → {output_path}")
    print(f"Skipped  : {result.skipped} records (no coordinates)")